# YOLOv8 License Plate Detection

This notebook reorganizes the license-plate detection training experiment from the project archive. It downloads a Roboflow dataset, trains YOLOv8, and validates the resulting detector.

**Security:** the Roboflow API key is read from `ROBOFLOW_API_KEY`; do not paste credentials into notebook cells or commit them.

In [ ]:
import os
from pathlib import Path

from roboflow import Roboflow
from ultralytics import YOLO

ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY")
if not ROBOFLOW_API_KEY:
    raise EnvironmentError("Set ROBOFLOW_API_KEY before downloading the dataset.")

## 1. Download the dataset

The original experiment used the `smart-ad-ezzae/plate-detect-oydq2` Roboflow project, version 3, exported in YOLOv8 format.

In [ ]:
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("smart-ad-ezzae").project("plate-detect-oydq2")
version = project.version(3)
dataset = version.download("yolov8")

DATA_YAML = str(Path(dataset.location) / "data.yaml")
print("Dataset:", dataset.location)
print("Config:", DATA_YAML)

## 2. Train YOLOv8

The following settings reflect the original experiment: YOLOv8-large, 100 epochs, and image size 256.

In [ ]:
model = YOLO("yolov8l.pt")

train_results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=256
)

## 3. Validate the best checkpoint

In [ ]:
BEST_WEIGHTS = Path("runs/detect/train/weights/best.pt")
if not BEST_WEIGHTS.exists():
    raise FileNotFoundError(
        f"Best checkpoint not found at {BEST_WEIGHTS}. "
        "Check the training run directory."
    )

best_model = YOLO(str(BEST_WEIGHTS))
validation_results = best_model.val(data=DATA_YAML, imgsz=256)

## 4. Run inference on an image

Replace the sample path with an image you are allowed to use.

In [ ]:
IMAGE_PATH = "sample_car.jpg"

if Path(IMAGE_PATH).is_file():
    predictions = best_model.predict(
        source=IMAGE_PATH,
        conf=0.25,
        save=True
    )
    print(predictions[0].boxes)
else:
    print(f"Add an image at {IMAGE_PATH} to run inference.")

## Limitations

This notebook trains an object detector that localizes license plates. It does not recognize the plate characters. Dataset access, training duration, and results depend on the Roboflow project version and runtime hardware.